In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd

In [2]:
# Variables path
layer_script = "block"

#process=""
disco="g"

# subj = "sub-A2004"


#subj = sys.argv[1] ## name of participant list

# Carpeta general
datadir = Path(f"{disco}:\MOUS_204")

#carpetas generales de datos
# mri_dir = datadir / f"{subj}"/"anat"
# meg_dir = datadir / f"{subj}"/"meg"

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"
channels_structure_path = output_preproc / "channels_structure"
preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)

output_analysis =output_preproc = datadir / "output_analysis"
analysis_path = output_analysis / f"analysis_{layer_script}"
analysis_path.mkdir(parents=True, exist_ok=True)

ACW_path = analysis_path / f"acw_{layer_script}"
ACW_path.mkdir(parents=True, exist_ok=True)

PLE_path = analysis_path / f"PLE_{layer_script}"
PLE_path.mkdir(parents=True, exist_ok=True)

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

<>:13: SyntaxWarning: invalid escape sequence '\M'
<>:13: SyntaxWarning: invalid escape sequence '\M'
C:\Users\UCM\AppData\Local\Temp\ipykernel_16372\2495930438.py:13: SyntaxWarning: invalid escape sequence '\M'
  datadir = Path(f"{disco}:\MOUS_204")


In [3]:
subj="sub-A2002"

In [4]:
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subj}_epochs_zinnen_{layer_script}-epo.fif")


FileNotFoundError: File does not exist: "g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2002_epochs_zinnen_block-epo.fif"

In [ ]:

#epochs
combinaciones = ["zinnen", "woorden"]

subj=[]
for subdirectorio in datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if subdirectorio.is_dir() and subdirectorio.name.startswith('sub-A2'):
        # Añadimos el nombre del sujeto a la lista
        subj.append(subdirectorio.name)
        print(subj)

subjects =subj[0:20]
subjects

##tablas de canales

###channels_path = datadir / "channels_structure" / f"channels_mag_{modality}.csv"

## this is the good one
##channels_path = channels_structure_path   / f"channels_mag_{modality}.csv"


channels = pd.read_csv(channels_path)
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]
channels_mag=channels_mag.tolist()
del channels

['sub-A2002']
['sub-A2002', 'sub-A2003']
['sub-A2002', 'sub-A2003', 'sub-A2004']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008', 'sub-A2009']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008', 'sub-A2009', 'sub-A2010']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008', 'sub-A2009', 'sub-A2010', 'sub-A2011']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008', 'sub-A2009', 'sub-A2010', 'sub-A2011', 'sub-A2013']
['sub-A2002', 'sub-A2003', 'sub-A2004', 'sub-A2005', 'sub-A2006', 'sub-A2007', 'sub-A2008', 'sub-A2009', 'sub-A2010', 'sub-A2011', 

In [12]:
len(channels_mag)

272

In [ ]:
# aplicar la acf EN epochs


from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd

def acf_epochs(subj, epochs,condition,adjusted=False,fft=True,alpha=None, bartlett_confint=True, missing="none", isplot=False): 
    
    #import epochs object and get data ONLY ON MEG DATA and ALSO EXC    data_epochs = None
    # if type(epochs) == "Epochs":
        data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data().get_data()

    #get duration
    duration= epochs.tmax - epochs.tmin
    #get sample frequency
    sfreq= epochs.info['sfreq']
    #get lags
    lags=duration*sfreq


    #list to store the values of acf, acw_50,acw_0 for each epoch for each sensor for each EPOCH
    acf_elect_all_epoch_all = []
    acf_mean_elect_all_epoch_all=[]

    #conjunto de todos los valores de ACW_0-ACW_50 para cada sensor en cada epoca
    acw_50_elect_all_epoch_all = []
    acw_0_elect_all_epoch_all = []


    #for each epoch
      
    for j in range(0,len(data_epochs)):
        #take data for each epoch
        data_epoch=data_epochs[j]

        #list to store the values of acf, acw_50,acw_0 for each SENSOR for each EPOCH

        acf_elect_all_epoch = []
        
        acf_mean_elect_all_epoch = []

        acw_50_elect_all_epoch = []
        acw_0_elect_all_epoch = []

    


        #calculus of value  for EACH SENSOR
        for i in range(0,len(data_epoch)):
            #lags se puede dejar por defecto porque te cogelos valores
            #hasta los que tiene sentido calcularlo, aunque yo voy a calcularlo para todos los lags
            #calculo de acf de sensor
            #alfa lo dejo endefault que es la confianza del 95% 

            acf_elect_epoch, qstat_vals_elect_epoch, pvals_elect_epoch = acf(
                data_epoch[i],
                adjusted=adjusted,
                fft=fft,
                qstat=True,
                nlags=lags,
                alpha=alpha,
                bartlett_confint=bartlett_confint,
                missing=missing
            )

            #conjunto de valores de acf en un electrodo en una epoca
            acf_elect_all_epoch.append(acf_elect_epoch)

            #promedio de valores de acf en un electrodo en una epoca
            acf_mean_elect_all_epoch.append(np.mean(acf_elect_epoch))


            #calculo de acw_50 y acw_0 de sensor for both in lags and in seconds
            #recuerda que acw_50 y acw_0 son valores unicos, no como acf que es un conjunto de valores
            acw_50_lags_elect_epoch = np.argmax(acf_elect_epoch <= 0.5)
            acw_50_s_elect_epoch = acw_50_lags_elect_epoch / sfreq

            acw_50_elect_all_epoch.append(acw_50_s_elect_epoch)


            acw_0_lags_elect_epoch= np.argmax(acf_elect_epoch <= 0)
            acw_0_s_elect_epoch = acw_0_lags_elect_epoch / sfreq

            acw_0_elect_all_epoch.append(acw_0_s_elect_epoch)


        #APPEND valores de TODOS LOS ELECTODOS en TODAS LAS EPOCAS

        acf_elect_all_epoch_all.append(acf_elect_all_epoch)
        acf_mean_elect_all_epoch_all.append(acf_mean_elect_all_epoch)

        acw_50_elect_all_epoch_all.append(acw_50_elect_all_epoch)
        acw_0_elect_all_epoch_all.append(acw_0_elect_all_epoch)


       

    #promedio de valores de acw_50 y acw_0 en un electrodo para todas las epocas
    # acw_50_elect_all_epoch_mean = []
    # acw_0_elect_all_epoch_mean = []


    # #promedio de los electrodos en todas las epocas
    # # Convertir la lista de listas en un array de NumPy para cálculos más eficientes
    # acw_50_elect_all_epoch_all_array = np.array(acw_50_elect_all_epoch_all)  # (21 filas, 59 columnas)
    # acw_0_elect_all_epoch_all_array = np.array(acw_0_elect_all_epoch_all)  # (21 filas, 59 columnas)

    # ##computo de la media de TODOS los electrodos en CADA EPOCA
    # acw_50_elect_mean_epoch_all = np.mean(acw_50_elect_all_epoch_all_array, axis=1)
    # acw_0_elect_mean_epoch_all = np.mean(acw_0_elect_all_epoch_all_array, axis=1)


    # ##computo de la media de cada los electrodos en TODAS EPOCAS
    # acw_50_elect_all_epoch_mean = np.mean(acw_50_elect_all_epoch_all_array, axis=0)       
    # acw_0_elect_all_epoch_mean = np.mean(acw_0_elect_all_epoch_all_array, axis=0)
    
    num_elects=len(channels_mag)
    num_epochs=len(data_epochs)
    shape_tabla=num_epochs*num_elects

    ##COMO ACF CONTIENE UNA SERIE TEMPORAL , sus dimensiones son (n_epochs, nchans, acf_series), Y quiero meterla con el resto de condiciones (nchans, n_epochs)
    #para que las dimensiones cuadren, la rehsape en n_epochs*nchans, acf_series, y lo transformo en una lista, para que cada fila sea una lista y así poder meterlo en el dataframe 

    acf_elect_all_epoch_all_list = np.array(acf_elect_all_epoch_all).reshape(num_epochs * num_elects, np.array(acf_elect_all_epoch_all).shape[2]).tolist()

    table_autocorrelation = pd.DataFrame({
        'Subject': [subj] * shape_tabla,  # Repite el sujeto para todas las filas
        'Condition': [condition] * shape_tabla,  # Repite la condición para todas las filas
        "Epoch": np.repeat(np.arange(num_epochs), num_elects),  # Repite épocas
        "Elect": np.tile(channels_mag, num_epochs),  # Número de electrodos
        "acf_elect_all_epoch_all": acf_elect_all_epoch_all_list,  # Convierte la tercera dimensión en listas
        "acw_50_elect_all_epoch_all": np.array(acw_50_elect_all_epoch_all).flatten(),
        "acw_0_elect_all_epoch_all": np.array(acw_0_elect_all_epoch_all).flatten()
    })
    #identificar acw_50


    #valores de estadística
        ## pero ver que valores son puto significativos de la ACF o de la ACW??



    isplot=False    

    # if isplot==True:
    #     # 1) Creamos un vector de lags en segundos,
    #     #    asumiendo que acf_val tiene tantos puntos como lags + 1.
    #     time_lags = np.arange(len(acf_val)) / sfreq

    #     # 2) Convertimos los índices (entero) de ACW_50 y ACW_0
    #     ACW_50_i = int(acw_50_lags)
    #     ACW_0_i = int(acw_0_lags)

    #     # 3) Figura
    #     plt.figure(figsize=(8, 4))

    #     # 4) Dibujamos la ACF en negro
    #     plt.plot(time_lags, acf_val, 'k', label='ACF')

    #     # 5) Acotamos el eje X a la duración total de la señal (o a lo que consideres)
    #     plt.xlim([0, time_lags[-1]])

    #     # 6) Rellenamos el área hasta ACW_50
    #     plt.fill_between(
    #         time_lags[:ACW_50_i + 1],
    #         acf_val[:ACW_50_i + 1],
    #         color='r', alpha=0.3, label='ACW-50 area'
    #     )

    #     # 7) Rellenamos el área hasta ACW_0
    #     plt.fill_between(
    #         time_lags[:ACW_0_i + 1],
    #         acf_val[:ACW_0_i + 1],
    #         color='m', alpha=0.3, label='ACW-0 area'
    #     )

    #     # 8) Título con valores de ACW
    #     plt.title(f'ACW-0 = {acw_0:.1f} s    ACW-50 = {acw_50:.1f} s')
    #     plt.xlabel('Lags (s)')
    #     plt.ylabel('Autocorrelation')
    #     plt.legend(loc='best')
    #     plt.show()

    return table_autocorrelation
    # return acf_epoch_all,acw_50_elect_all_epoch_all, acw_0_elect_all_epoch_all




In [7]:
table_autocorrelation= acf_epochs(subjects[0],epochs_zinnen, condition="zinnen", isplot=False)

C:\Users\UCM\AppData\Local\Temp\ipykernel_9928\3687354282.py:12: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


In [8]:
subjects

['sub-A2002',
 'sub-A2003',
 'sub-A2004',
 'sub-A2005',
 'sub-A2006',
 'sub-A2007',
 'sub-A2008',
 'sub-A2009',
 'sub-A2010',
 'sub-A2011',
 'sub-A2013',
 'sub-A2014',
 'sub-A2015',
 'sub-A2016',
 'sub-A2017',
 'sub-A2019',
 'sub-A2020',
 'sub-A2021',
 'sub-A2024',
 'sub-A2025']

In [9]:
combinaciones

['zinnen', 'woorden']

In [10]:
##codigo para agrupar todas las tablas

all_tables = []

for i in range(0,len(subjects)):
    for h in range(0,len(combinaciones)):
        try:
            subj=subjects[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subj}_epochs_{combinacion}_{layer_script}-epo.fif"
            epochs = mne.read_epochs(path_epochs)
            table_autocorrelation= acf_epochs(subj, epochs, condition=combinacion,isplot=False)
            all_tables.append(table_autocorrelation)
            del epochs
        except:
            print(f"Error en {subj} {combinacion}")

autocorrelation_subjects_all = pd.concat(all_tables, ignore_index=True)


Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2002_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
17 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_9928\3687354282.py:12: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2002_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
22 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_9928\3687354282.py:12: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2003_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
20 matching events found
No baseline correction applied
0 projection items activated


...

Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2007_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
18 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_9928\3687354282.py:12: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2007_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_9928\3687354282.py:12: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Error en sub-A2008 zinnen
Error en sub-A2008 woorden
Error en sub-A2009 zinnen
Error en sub-A2009 woorden
Error en sub-A2010 zinnen
Error en sub-A2010 woorden
Error en sub-A2011 zinnen
Error en sub-A2011 woorden
Error en sub-A2013 zinnen
Error en sub-A2013 woorden
Error en sub-A2014 zinnen
Error en sub-A2014 woorden
Error en sub-A2015 zinnen
Error en sub-A2015 woorden
Error en sub-A2016 zinnen
Error en sub-A2016 woorden
Error en sub-A2017 zinnen
Error en sub-A2017 woorden
Error en sub-A2019 zinnen
Error en sub-A2019 woorden
Error en sub-A2020 zinnen
Error en sub-A2020 woorden
Error en sub-A2021 zinnen
Error en sub-A2021 woorden
Error en sub-A2024 zinnen
Error en sub-A2024 woorden
Error en sub-A2025 zinnen
Error en sub-A2025 woorden


In [ ]:
autocorrelation_subjects_all

In [ ]:
ACW_path

In [ ]:
autocorrelation_subjects_all.to_pickle(ACW_path / f"autocorrelation_subjects_all.pickle")

In [ ]:
# Lista de variables a analizar
variables = [
    acf_elect_all_epoch_all,
    acf_mean_elect_all_epoch_all,
    acw_50_elect_all_epoch_all,
    acw_0_elect_all_epoch_all,
    acw_50_elect_mean_epoch_all,
    acw_0_elect_mean_epoch_all,
    acw_50_elect_all_epoch_mean,
    acw_0_elect_all_epoch_mean
]

# Lista de nombres para imprimir
variable_names = [
    "acf_elect_all_epoch_all",
    "acf_mean_elect_all_epoch_all",
    "acw_50_elect_all_epoch_all",
    "acw_0_elect_all_epoch_all",
    "acw_50_elect_mean_epoch_all",
    "acw_0_elect_mean_epoch_all",
    "acw_50_elect_all_epoch_mean",
    "acw_0_elect_all_epoch_mean"
]

# Bucle para imprimir los tipos y dimensiones
for var, name in zip(variables, variable_names):
    print(f"\n{name}:")
    print(f"  type({name}) is {type(var)}")
    
    try:
        print(f"  type({name}[0]) is {type(var[0])}")
        
        try:
            print(f"  type({name}[0][0]) is {type(var[0][0])}")
        except (IndexError, TypeError):
            print(f"  {name}[0] does not have a second dimension.")
            try:
                print(f"  type({name}[0][0][0]) is {type(var[0][0][0])}")
            except (IndexError, TypeError):
                print(f"  {name}[0][0] does not have a third dimension.")
    
    except (IndexError, TypeError):
        print(f"  {name} does not have a first dimension.")

# Lista de variables a analizar
variables = [
    acf_elect_all_epoch_all,
    acf_mean_elect_all_epoch_all,
    acw_50_elect_all_epoch_all,
    acw_0_elect_all_epoch_all,
    acw_50_elect_mean_epoch_all,
    acw_0_elect_mean_epoch_all,
    acw_50_elect_all_epoch_mean,
    acw_0_elect_all_epoch_mean
]

# Lista de nombres para imprimir
variable_names = [
    "acf_elect_all_epoch_all",
    "acf_mean_elect_all_epoch_all",
    "acw_50_elect_all_epoch_all",
    "acw_0_elect_all_epoch_all",
    "acw_50_elect_mean_epoch_all",
    "acw_0_elect_mean_epoch_all",
    "acw_50_elect_all_epoch_mean",
    "acw_0_elect_all_epoch_mean"
]

# Bucle para imprimir los tipos y dimensiones
for var, name in zip(variables, variable_names):
    print(f"\n{name}:")
    print(f"  type({name}) is {type(var)}")
    
    try:
        print(f"  type({name}[0]) is {type(var[0])}")
        
        try:
            print(f"  type({name}[0][0]) is {type(var[0][0])}")
        except (IndexError, TypeError):
            print(f"  {name}[0] does not have a second dimension.")
            try:
                print(f"  type({name}[0][0][0]) is {type(var[0][0][0])}")
            except (IndexError, TypeError):
                print(f"  {name}[0][0] does not have a third dimension.")
    
    except (IndexError, TypeError):
        print(f"  {name} does not have a first dimension.")